# 07b | Linear regression: fitting in Python

Python has two libraries for regression, with different roles:

| | `statsmodels` | `scikit-learn` |
|---|---|---|
| Like R's | `lm()` + `summary()` | no direct equivalent |
| Purpose | **Inference**: coefficients, standard errors, p-values, diagnostics | **Prediction**: fit, predict, cross-validate |
| Intercept | added automatically by the formula interface | added automatically |

Statistics interviews and take-homes use `statsmodels`; ML pipelines use `scikit-learn` (lesson 10).

The `statsmodels` formula interface is almost identical to R's: `smf.ols("y ~ x1 + x2", data=df).fit()`.

About 55 minutes.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(0)
n = 300
df = pd.DataFrame({"x1": rng.normal(size=n), "x2": rng.normal(size=n)})
df["group"] = rng.choice(["a", "b", "c"], size=n)
df["y"] = 1 + 2 * df["x1"] - 1 * df["x2"] + rng.normal(scale=2, size=n)

model = smf.ols("y ~ x1 + x2", data=df).fit()
print(model.summary().tables[1])       # coefficient table; drop .tables[1] for the full output
print("R2 =", round(model.rsquared, 3))

Common ways to pull out results (R's `coef(m)`, `confint(m)`, `predict(m)`):

| Want | Code |
|---|---|
| Coefficients | `model.params` (a Series; `model.params["x1"]`) |
| Standard errors / p-values | `model.bse` / `model.pvalues` |
| Confidence intervals | `model.conf_int()` |
| R² / adjusted R² | `model.rsquared` / `model.rsquared_adj` |
| Residuals / fitted values | `model.resid` / `model.fittedvalues` |
| Predictions | `model.predict(new_df)` |

The formula syntax is the same as R's: `x1 * x2` gives main effects plus the interaction, `x1:x2` only the interaction, `C(group)` treats a column as categorical, and `np.log(x)` can go straight into the formula.

### ✏️ Exercise 1 (Write it yourself)

Fit `y ~ x1 + x2 + C(group)` on `df` and store the result in `m2`. Then extract:

- `b1`: the coefficient on `x1`;
- `p1`: the p-value for `x1`;
- `ci_low`, `ci_high`: the lower and upper bounds of the 95% confidence interval for the `x1` coefficient.

`model.conf_int()` returns a table whose row labels are variable names and whose two columns are named `0` and `1`; use `.loc["x1", 0]` to get a value.

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
m2 = smf.ols("y ~ x1 + x2 + C(group)", data=df).fit()
b1 = m2.params["x1"]
p1 = m2.pvalues["x1"]
ci = m2.conf_int()
ci_low, ci_high = ci.loc["x1", 0], ci.loc["x1", 1]
print(b1, p1, ci_low, ci_high)
```

</details>

## 1. OLS by hand

An interviewer may ask you to implement $\hat\beta = (X^\top X)^{-1} X^\top y$ directly in NumPy. Two key points:

- You have to add the column of ones for the intercept yourself;
- **Don't actually invert the matrix.** Write `np.linalg.solve(X.T @ X, X.T @ y)` (solving the linear system), which is numerically more stable and faster. Mentioning this earns points.

`@` is matrix multiplication and `.T` is transpose.

### ✏️ Exercise 2 (Fill in the blanks)

Use the normal equations to compute the coefficients `beta` (in order: intercept, x1, x2).

In [ ]:
X = np.column_stack([np.ones(len(df)), df["x1"], df["x2"]])     # first column is the intercept
y = df["y"].to_numpy()
beta = np.linalg.solve(X.T @ ____, X.T @ ____)
print(beta.round(4))

<details>
<summary><b>Answer</b></summary>

```python
X = np.column_stack([np.ones(len(df)), df["x1"], df["x2"]])     # first column is the intercept
y = df["y"].to_numpy()
beta = np.linalg.solve(X.T @ X, X.T @ y)
print(beta.round(4))
```

</details>

### ✏️ Exercise 3 (Write it yourself)

Write a function `r_squared(y, y_hat)` that returns $R^2 = 1 - \dfrac{\sum (y - \hat y)^2}{\sum (y - \bar y)^2}$. The inputs are two NumPy arrays.

In [ ]:
def r_squared(y, y_hat):
    # Write your code below
    pass

<details>
<summary><b>Answer</b></summary>

```python
def r_squared(y, y_hat):
    sse = ((y - y_hat) ** 2).sum()
    sst = ((y - y.mean()) ** 2).sum()
    return 1 - sse / sst
```

</details>

## 2. Heteroskedasticity and robust standard errors

Below we generate data whose error grows with x. Adding `cov_type="HC3"` inside `.fit()` gives heteroskedasticity-robust standard errors. **The coefficients do not change at all; only the standard errors do.**

In [ ]:
rng = np.random.default_rng(1)
n = 400
h = pd.DataFrame({"x": rng.uniform(0, 10, n)})
h["y"] = 2 + 0.5 * h["x"] + rng.normal(scale=0.3 * h["x"] ** 1.5)      # noise grows with x

classic = smf.ols("y ~ x", data=h).fit()
print("coef:", classic.params["x"].round(4), " classical SE:", classic.bse["x"].round(4))

### ✏️ Exercise 4 (Modify)

Refit the same data, this time with HC3 robust standard errors, and store the result in `robust`. Then store the robust standard error of `x` in `se_robust`.

In [ ]:
robust = smf.ols("y ~ x", data=h).fit()
se_robust = robust.bse["x"]
print(robust.params["x"].round(4), se_robust.round(4))

<details>
<summary><b>Answer</b></summary>

```python
robust = smf.ols("y ~ x", data=h).fit(cov_type="HC3")
se_robust = robust.bse["x"]
print(robust.params["x"].round(4), se_robust.round(4))
```

Here the robust standard error is larger than the classical one: the classical formula underestimates the uncertainty and makes you overconfident. In applied work, many people report robust standard errors by default.

</details>

## 3. Multicollinearity and VIF

In the data below `x2` is almost a copy of `x1`. Look at two things: the standard errors of the two coefficients become large, and the VIFs are high.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

rng = np.random.default_rng(2)
n = 200
c = pd.DataFrame({"x1": rng.normal(size=n)})
c["x2"] = c["x1"] + rng.normal(scale=0.1, size=n)       # highly correlated with x1
c["x3"] = rng.normal(size=n)
c["y"] = 1 + c["x1"] + c["x2"] + c["x3"] + rng.normal(size=n)

print(smf.ols("y ~ x1 + x2 + x3", data=c).fit().bse.round(3))

Xc = np.column_stack([np.ones(n), c["x1"], c["x2"], c["x3"]])
print([round(variance_inflation_factor(Xc, j), 1) for j in [1, 2, 3]])   # skip column 0 (the intercept)

### ✏️ Exercise 5 (Write it yourself)

Compute the VIF of `x1` from the definition: regress `x1` on `x2` and `x3` to get $R^2$, then compute $\mathrm{VIF} = \dfrac{1}{1 - R^2}$. Store the result in `vif_x1`.

In [ ]:
# Write your code below

<details>
<summary><b>Answer</b></summary>

```python
aux = smf.ols("x1 ~ x2 + x3", data=c).fit()
vif_x1 = 1 / (1 - aux.rsquared)
print(vif_x1)
```

</details>

## 4. Omitted variable bias: see it for yourself

In the true model, `ability` affects both `educ` and `wage`. Compare the coefficient on `educ` with and without controlling for `ability`.

In [ ]:
rng = np.random.default_rng(3)
n = 5000
w = pd.DataFrame({"ability": rng.normal(size=n)})
w["educ"] = 12 + 2 * w["ability"] + rng.normal(size=n)                    # higher-ability people get more years of education
w["wage"] = 5 + 1.0 * w["educ"] + 3.0 * w["ability"] + rng.normal(size=n)  # the true effect of education is 1.0

print("controlling for ability:", smf.ols("wage ~ educ + ability", data=w).fit().params["educ"].round(3))
print("without ability:        ", smf.ols("wage ~ educ", data=w).fit().params["educ"].round(3))

**🗣 Say it out loud:** Without controlling for ability, the coefficient on educ above is about 2.2, while the true value is 1.0. Explain this number with the omitted variable bias formula.

<details>
<summary><b>Reference answer</b></summary>

Bias = γ × δ. γ is the effect of ability on wage, 3.0. δ is the slope from regressing ability on educ: $\mathrm{Cov}(\text{ability}, \text{educ})/\mathrm{Var}(\text{educ}) = 2/(4 + 1) = 0.4$. So the bias = 3.0 × 0.4 = 1.2, and the expected coefficient is 1.0 + 1.2 = 2.2, matching the simulation.

</details>

## Summary

- `smf.ols("y ~ x1 + x2", data=df).fit()`, the same formulas as R's `lm`;
- `.params`, `.bse`, `.pvalues`, `.conf_int()`, `.rsquared`;
- Robust standard errors: `.fit(cov_type="HC3")`;
- For OLS by hand use `np.linalg.solve`, not a matrix inverse.

On to **08a**.